<a href="https://colab.research.google.com/github/Navya-Mittal/Recommender_System/blob/main/recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributed Hybrid Recommender System

**Stack:** PySpark MLlib, ALS Collaborative Filtering, Sentence-Transformers, KMeans, PCA, Plotly

---

This notebook implements a distributed hybrid recommendation engine using Apache Spark's MLlib. The pipeline consists of two complementary components:

- **Collaborative Filtering (ALS):** Learns latent user and item factor vectors from historical interaction data. Model selection is performed via cross-validated RMSE optimisation.
- **Content-Based Filtering:** Encodes product text into dense semantic vectors using a local sentence embedding model, then groups products into clusters via KMeans.
- **Hybrid Fusion:** Combines both signals using a weighted score, providing personalised recommendations for active users while maintaining meaningful fallback behaviour for cold-start cases.

All embeddings are generated locally using `sentence-transformers`. No external API key is required.

---

## Table of Contents

1. Environment Setup
2. Data Loading and Synthetic Interaction Generation
3. Collaborative Filtering with ALS
4. Model Evaluation
5. Hyperparameter Tuning via CrossValidator
6. Content-Based Filtering with Sentence Embeddings
7. Elbow Method for Cluster Selection
8. Hybrid Recommender Fusion
9. Cluster Visualisation via PCA
10. End-to-End Recommendation Demo
11. Architecture and Design Notes


---
## 1. Environment Setup

In [1]:
!pip install pyspark sentence-transformers plotly --quiet

In [2]:
import os
import random
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sentence_transformers import SentenceTransformer

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans

print('All imports successful.')

All imports successful.


In [3]:
# Initialise a local Spark session.
# In a cluster environment, remove .master('local[*]') and configure externally.
spark = (
    SparkSession.builder
    .appName('HybridRecommenderSystem')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} session initialised.')

Spark 4.0.2 session initialised.


---
## 2. Data Loading and Synthetic Interaction Generation

The product catalog is loaded from `products_dataset.csv`. Since no real user interaction logs are available, a synthetic dataset of explicit ratings is generated to simulate production conditions. Each user is assigned a random subset of products with ratings sampled from a skewed distribution, reflecting the tendency of real users to rate positively.

In [4]:
products_pd = pd.read_csv('products_dataset.csv')
print(f'Loaded {len(products_pd):,} products.')
products_pd.head(3)

Loaded 2,000 products.


,product_id,title,description
0,P0,Men's 3X Large Carbon Heather Cotton/Polyester...,"This heavyweight, water-repellent hooded sweat..."
1,P1,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...,If you need more length between your existing ...
2,P2,Large Tapestry Bolster Bed,Polyester cover resembling rich Italian tapest...


In [5]:
# ALS requires integer user and item identifiers.
# Map the string product_id column to a zero-based integer index.
products_pd['item_idx'] = products_pd.index
id_to_idx = dict(zip(products_pd['product_id'], products_pd['item_idx']))
idx_to_id = dict(zip(products_pd['item_idx'], products_pd['product_id']))

NUM_PRODUCTS          = len(products_pd)
NUM_USERS             = 500
INTERACTIONS_PER_USER = (5, 40)

random.seed(42)
np.random.seed(42)

records = []
for user_id in range(NUM_USERS):
    n     = random.randint(*INTERACTIONS_PER_USER)
    items = random.sample(range(NUM_PRODUCTS), n)
    for item_idx in items:
        # Sample a base rating, then add Gaussian noise to simulate real variation
        base   = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30])
        rating = float(np.clip(base + np.random.normal(0, 0.3), 1, 5))
        records.append((user_id, item_idx, round(rating, 1)))

interactions_pd = pd.DataFrame(records, columns=['user_id', 'item_idx', 'rating'])
print(f'Generated {len(interactions_pd):,} interactions '
      f'({NUM_USERS} users, average {len(interactions_pd) // NUM_USERS} interactions each).')
interactions_pd['rating'].describe().round(2)

Generated 11,207 interactions (500 users, average 22 interactions each).


,rating
count,11207.00
mean,3.71
std,1.11
min,1.00
25%,3.00
50%,3.90
75%,4.70
max,5.00


In [6]:
# Define an explicit schema to avoid Spark type inference overhead
schema = StructType([
    StructField('user_id',  IntegerType(), False),
    StructField('item_idx', IntegerType(), False),
    StructField('rating',   FloatType(),   False),
])

interactions_df = spark.createDataFrame(interactions_pd, schema=schema)
interactions_df.show(5)

# Standard 80/20 train-test split with a fixed seed for reproducibility
train_df, test_df = interactions_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train set: {train_df.count():,} rows  |  Test set: {test_df.count():,} rows')

+-------+--------+------+
|user_id|item_idx|rating|
+-------+--------+------+
|      0|      51|   3.7|
|      0|    1518|   3.1|
|      0|     563|   2.1|
|      0|     501|   5.0|
|      0|     457|   1.0|
+-------+--------+------+
only showing top 5 rows
Train set: 9,019 rows  |  Test set: 2,188 rows


---
## 3. Collaborative Filtering — ALS

The ALS algorithm factorises the sparse user-item rating matrix **R** into two dense latent-factor matrices:

$$R \approx U \cdot V^T \quad (U \in \mathbb{R}^{|users| \times k},\; V \in \mathbb{R}^{|items| \times k})$$

The algorithm alternates between holding **U** fixed and solving for **V**, then vice versa. Each sub-problem reduces to a regularised least-squares solve, which Spark executes in parallel across partitions. This design scales horizontally to billions of interactions without modification.

Key parameters:
- `rank`: number of latent factors — controls model capacity
- `regParam`: L2 regularisation weight — controls overfitting
- `coldStartStrategy='drop'`: excludes users and items absent from training when computing evaluation metrics

In [7]:
als = ALS(
    maxIter=15,
    rank=50,
    regParam=0.1,
    userCol='user_id',
    itemCol='item_idx',
    ratingCol='rating',
    coldStartStrategy='drop',
    nonnegative=False,
    implicitPrefs=False,  # set to True when using implicit signals such as clicks or views
    seed=42
)

als_model = als.fit(train_df)
print('Baseline ALS model trained.')

Baseline ALS model trained.


---
## 4. Model Evaluation

The model is evaluated on the held-out test set using Root Mean Square Error (RMSE) and Mean Absolute Error (MAE). A predicted-vs-actual scatter plot provides a visual diagnostic of systematic bias or variance in the predictions.

In [8]:
predictions = als_model.transform(test_df)

rmse_eval = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
mae_eval  = RegressionEvaluator(metricName='mae',  labelCol='rating', predictionCol='prediction')

baseline_rmse = rmse_eval.evaluate(predictions)
baseline_mae  = mae_eval.evaluate(predictions)

print('Baseline ALS Performance')
print(f'  RMSE : {baseline_rmse:.4f}')
print(f'  MAE  : {baseline_mae:.4f}')

Baseline ALS Performance
  RMSE : 1.4821
  MAE  : 1.2465


In [9]:
# Sample 30% of predictions for plotting to keep the chart readable
pred_pd = predictions.sample(False, 0.3, seed=42).toPandas()

fig = px.scatter(
    pred_pd,
    x='rating',
    y='prediction',
    opacity=0.4,
    trendline='ols',
    title=f'Predicted vs Actual Ratings  (RMSE = {baseline_rmse:.4f})',
    labels={'rating': 'Actual Rating', 'prediction': 'Predicted Rating'}
)
# Diagonal reference line representing a perfect predictor
fig.add_shape(
    type='line', x0=1, y0=1, x1=5, y1=5,
    line=dict(color='red', dash='dash')
)
fig.update_layout(width=650, height=450)
fig.show()

---
## 5. Hyperparameter Tuning via CrossValidator

A grid search over `rank` and `regParam` is conducted using Spark's `CrossValidator` with 2-fold cross-validation. The search space is deliberately limited to keep wall-clock time reasonable on a single-node environment. The configuration yielding the lowest average validation RMSE is selected as the final model.

In [10]:
# Grid: 6 configurations x 2 folds = 12 total fits
als_cv = ALS(
    userCol='user_id',
    itemCol='item_idx',
    ratingCol='rating',
    coldStartStrategy='drop',
    seed=42
)

param_grid = (
    ParamGridBuilder()
    .addGrid(als_cv.rank,     [20, 50])
    .addGrid(als_cv.regParam, [0.01, 0.1, 0.5])
    .build()
)

cv = CrossValidator(
    estimator=als_cv,
    estimatorParamMaps=param_grid,
    evaluator=rmse_eval,
    numFolds=2,
    parallelism=4,  # number of folds evaluated concurrently
    seed=42
)

print(f'Running {len(param_grid)} configurations x 2 folds = {len(param_grid) * 2} fits...')
cv_model   = cv.fit(train_df)
best_model = cv_model.bestModel

tuned_rmse = rmse_eval.evaluate(best_model.transform(test_df))
print(f'Tuned RMSE  : {tuned_rmse:.4f}  (baseline: {baseline_rmse:.4f})')
print(f'Best rank   : {best_model.rank}')
print(f'Improvement : {((baseline_rmse - tuned_rmse) / baseline_rmse * 100):.1f}%')

Running 6 configurations x 2 folds = 12 fits...
Tuned RMSE  : 1.4064  (baseline: 1.4821)
Best rank   : 50
Improvement : 5.1%


In [11]:
# Visualise average RMSE across all evaluated configurations
param_labels = [
    f"rank={p[als_cv.rank]}, regParam={p[als_cv.regParam]}"
    for p in param_grid
]
cv_df = (
    pd.DataFrame({'configuration': param_labels, 'RMSE': cv_model.avgMetrics})
    .sort_values('RMSE')
)

fig = px.bar(
    cv_df,
    x='RMSE',
    y='configuration',
    orientation='h',
    color='RMSE',
    color_continuous_scale='RdYlGn_r',
    title='Cross-Validation RMSE by Hyperparameter Configuration'
)
fig.update_layout(width=820, height=420, yaxis_title='')
fig.show()

---
## 6. Content-Based Filtering with Sentence Embeddings

Content-based filtering operates independently of user history, making it suitable for cold-start scenarios. Each product's title and description are concatenated and encoded into a 384-dimensional vector using `all-MiniLM-L6-v2`, a lightweight sentence transformer model from HuggingFace.

The model is downloaded once (~80 MB) and cached locally. L2 normalisation is applied so that Euclidean distance in the embedding space corresponds to cosine dissimilarity, which is more appropriate for clustering semantically related text.

Products are then grouped into clusters using Spark's distributed KMeans implementation. Items within the same cluster serve as content-based recommendation candidates.

In [12]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded. Embedding dimension: {encoder.get_sentence_embedding_dimension()}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384


In [13]:
# Combine title and description into a single text field per product
products_pd['combined_text'] = (
    products_pd['title'].fillna('') + ' ' +
    products_pd['description'].fillna('')
)
texts = products_pd['combined_text'].tolist()
print(f'Encoding {len(texts):,} product descriptions...')

# batch_size=64 is appropriate for CPU; increase to 256 when a GPU is available
embedding_vectors = encoder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

EMB_DIM = embedding_vectors.shape[1]
print(f'Encoding complete. Output shape: {embedding_vectors.shape}')

Encoding 2,000 product descriptions...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Encoding complete. Output shape: (2000, 384)


In [14]:
# Each embedding dimension becomes a separate column so VectorAssembler can
# construct the DenseVector required by Spark ML
emb_col_names = [f'emb_{i}' for i in range(EMB_DIM)]

emb_pd = pd.DataFrame(embedding_vectors, columns=emb_col_names)
emb_pd['product_id'] = products_pd['product_id'].values
emb_pd['title']      = products_pd['title'].values
emb_pd['item_idx']   = products_pd['item_idx'].values

emb_spark = spark.createDataFrame(emb_pd)
assembler = VectorAssembler(inputCols=emb_col_names, outputCol='features')
emb_spark = assembler.transform(emb_spark).select('product_id', 'title', 'item_idx', 'features')

print(f'Embedding DataFrame constructed. Rows: {emb_spark.count():,}, Dimensions: {EMB_DIM}')

Embedding DataFrame constructed. Rows: 2,000, Dimensions: 384


In [15]:
NUM_CLUSTERS = 8

kmeans = KMeans(
    k=NUM_CLUSTERS,
    seed=42,
    featuresCol='features',
    predictionCol='cluster',
    maxIter=30,
    tol=1e-4
)
kmeans_model  = kmeans.fit(emb_spark)
clustered_emb = kmeans_model.transform(emb_spark)

print(f'KMeans complete. Within-cluster sum of squared errors (WSSSE): {kmeans_model.summary.trainingCost:,.2f}')
clustered_emb.groupBy('cluster').count().orderBy('cluster').show()

KMeans complete. Within-cluster sum of squared errors (WSSSE): 1,256.93
+-------+-----+
|cluster|count|
+-------+-----+
|      0|  190|
|      1|  360|
|      2|  262|
|      3|  337|
|      4|  177|
|      5|  316|
|      6|  170|
|      7|  188|
+-------+-----+



---
## 7. Elbow Method for Cluster Selection

The optimal number of clusters is estimated by plotting WSSSE against increasing values of k. The point at which the rate of reduction begins to diminish — the elbow — indicates a reasonable trade-off between cluster compactness and model complexity.

In [16]:
wssse_scores = []
k_range = range(2, 15)

for k in k_range:
    m = KMeans(
        k=k, seed=42,
        featuresCol='features',
        predictionCol='cluster',
        maxIter=20
    ).fit(emb_spark)
    wssse_scores.append(m.summary.trainingCost)
    print(f'  k={k:2d}  WSSSE={m.summary.trainingCost:,.1f}')

elbow_df = pd.DataFrame({'k': list(k_range), 'WSSSE': wssse_scores})

fig = px.line(
    elbow_df, x='k', y='WSSSE', markers=True,
    title='Elbow Method: WSSSE vs Number of Clusters',
    labels={'k': 'Number of Clusters (k)', 'WSSSE': 'Within-Cluster Sum of Squares'}
)
fig.update_layout(width=650, height=420)
fig.show()

  k= 2  WSSSE=1,549.8
  k= 3  WSSSE=1,387.9
  k= 4  WSSSE=1,354.6
  k= 5  WSSSE=1,318.0
  k= 6  WSSSE=1,291.6
  k= 7  WSSSE=1,281.9
  k= 8  WSSSE=1,257.0
  k= 9  WSSSE=1,247.2
  k=10  WSSSE=1,224.7
  k=11  WSSSE=1,206.4
  k=12  WSSSE=1,203.6
  k=13  WSSSE=1,183.1
  k=14  WSSSE=1,174.7


---
## 8. Hybrid Recommender Fusion

The two recommendation signals are combined using a linear weighted blend:

$$\text{score}(u, i) = \alpha \cdot \text{score}_{\text{ALS}}(u, i) + (1 - \alpha) \cdot \text{score}_{\text{content}}(i)$$

Both signals are normalised to the range [0, 1] before blending to ensure commensurability. The `ALPHA` parameter controls the relative contribution of each signal. A value of 0.7 biases recommendations toward personalised ALS output while retaining a meaningful content-based component for items outside the ALS candidate set.

In [17]:
TOP_N = 10

# Generate top-N ALS candidates for every user in the training set
als_recs = best_model.recommendForAllUsers(TOP_N)

# Explode the recommendations struct array into individual rows
als_recs_flat = (
    als_recs
    .select('user_id', F.explode('recommendations').alias('rec'))
    .select(
        'user_id',
        F.col('rec.item_idx').alias('item_idx'),
        F.col('rec.rating').alias('als_score')
    )
)

# Min-max normalise ALS scores to [0, 1] for comparability with content scores
max_s = als_recs_flat.agg(F.max('als_score')).collect()[0][0]
min_s = als_recs_flat.agg(F.min('als_score')).collect()[0][0]
als_recs_flat = als_recs_flat.withColumn(
    'als_score_norm',
    (F.col('als_score') - min_s) / (max_s - min_s)
)

print('ALS scores normalised to [0, 1].')
als_recs_flat.show(5)

ALS scores normalised to [0, 1].
+-------+--------+---------+-------------------+
|user_id|item_idx|als_score|     als_score_norm|
+-------+--------+---------+-------------------+
|      0|     688|3.8057652|   0.37694690681678|
|      0|    1416|3.6097636| 0.2878626319158998|
|      0|     512|3.5818791| 0.2751889095067872|
|      0|    1471|3.5763175| 0.2726611219402035|
|      0|    1867|3.5739722|0.27159515347908614|
+-------+--------+---------+-------------------+
only showing top 5 rows


In [18]:
# Collect cluster assignments to the driver for lightweight lookup
cluster_pd = clustered_emb.select('product_id', 'title', 'item_idx', 'cluster').toPandas()


def content_based_recommendations(product_id, n=TOP_N):
    """
    Return up to n product IDs from the same cluster as the given product.
    Returns an empty list if the product is not found in the catalog.
    """
    row = cluster_pd.loc[cluster_pd['product_id'] == product_id, 'cluster']
    if row.empty:
        return []
    cluster_id = row.values[0]
    candidates = cluster_pd[
        (cluster_pd['cluster'] == cluster_id) &
        (cluster_pd['product_id'] != product_id)
    ]
    return candidates.sample(min(n, len(candidates)), random_state=42)['product_id'].tolist()


ALPHA = 0.7  # relative weight assigned to the ALS signal


def hybrid_recommend(user_id, recently_viewed_ids, alpha=ALPHA, n=TOP_N):
    """
    Generate top-n hybrid recommendations for a user.

    Parameters
    ----------
    user_id             : int   — identifier of the target user
    recently_viewed_ids : list  — product_id strings recently viewed by the user
    alpha               : float — weight applied to the ALS signal (1 - alpha applied to content)
    n                   : int   — number of recommendations to return

    Returns
    -------
    pandas.DataFrame sorted by hybrid_score descending
    """
    # Retrieve ALS candidates for this user
    user_als = als_recs_flat.filter(F.col('user_id') == user_id).toPandas()
    als_dict = dict(zip(user_als['item_idx'], user_als['als_score_norm']))

    # Build content scores by aggregating cluster co-occurrence across recently viewed products.
    # Items appearing in multiple cluster neighbourhoods receive higher scores.
    cb_scores = {}
    for pid in recently_viewed_ids:
        for rec_pid in content_based_recommendations(pid, n=20):
            idx = id_to_idx.get(rec_pid)
            if idx is not None:
                cb_scores[idx] = cb_scores.get(idx, 0) + 1

    if cb_scores:
        max_cb    = max(cb_scores.values())
        cb_scores = {k: v / max_cb for k, v in cb_scores.items()}

    # Exclude items the user has already viewed
    viewed_idx = {id_to_idx[p] for p in recently_viewed_ids if p in id_to_idx}
    all_items  = (set(als_dict) | set(cb_scores)) - viewed_idx

    rows = []
    for idx in all_items:
        als_s  = als_dict.get(idx, 0.0)
        cb_s   = cb_scores.get(idx, 0.0)
        hybrid = alpha * als_s + (1 - alpha) * cb_s
        pid    = idx_to_id.get(idx, 'unknown')
        title  = products_pd.loc[products_pd['product_id'] == pid, 'title'].values
        title  = title[0] if len(title) else 'N/A'
        rows.append({
            'product_id':   pid,
            'item_idx':     idx,
            'als_score':    als_s,
            'cb_score':     cb_s,
            'hybrid_score': hybrid,
            'title':        title
        })

    return pd.DataFrame(rows).sort_values('hybrid_score', ascending=False).head(n)


print('Hybrid recommender defined.')

Hybrid recommender defined.


---
## 9. Cluster Visualisation via PCA

Principal Component Analysis reduces the 384-dimensional embedding space to two dimensions for visual inspection. The resulting scatter plot allows qualitative assessment of cluster separation and can surface cases where semantically dissimilar products have been grouped together.

In [19]:
pca_model  = PCA(k=2, inputCol='features', outputCol='pcaFeatures').fit(clustered_emb)
pca_result = pca_model.transform(clustered_emb)

ev = pca_model.explainedVariance
print(f'Explained variance — PC1: {ev[0]:.2%}  |  PC2: {ev[1]:.2%}  |  Total: {sum(ev):.2%}')

Explained variance — PC1: 11.28%  |  PC2: 5.20%  |  Total: 16.48%


In [20]:
pca_pd = pca_result.select('product_id', 'title', 'cluster', 'pcaFeatures').toPandas()
pca_pd['x']           = pca_pd['pcaFeatures'].apply(lambda v: float(v[0]))
pca_pd['y']           = pca_pd['pcaFeatures'].apply(lambda v: float(v[1]))
pca_pd['cluster_str'] = pca_pd['cluster'].astype(str)
pca_pd['short_title'] = pca_pd['title'].str[:55]

fig = px.scatter(
    pca_pd,
    x='x', y='y',
    color='cluster_str',
    opacity=0.65,
    hover_data={'product_id': True, 'short_title': True, 'cluster': True, 'x': False, 'y': False},
    title=(
        f'Product Embedding Clusters — PCA Projection ({NUM_CLUSTERS} clusters)<br>'
        f'<sup>Explained variance: PC1={ev[0]:.1%}, PC2={ev[1]:.1%}</sup>'
    ),
    labels={'x': 'Principal Component 1', 'y': 'Principal Component 2', 'cluster_str': 'Cluster'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(width=820, height=600)
fig.show()

---
## 10. End-to-End Recommendation Demo

The full pipeline is demonstrated for a single target user. Recently viewed products are used to seed the content-based signal, and the hybrid function returns a ranked list of recommendations with individual score components for interpretability.

In [21]:
TARGET_USER              = 42
recently_viewed_products = ['P316', 'P333', 'P1115', 'P1691', 'P1082', 'P397', 'P1441', 'P1054']

print('-' * 70)
print(f'User {TARGET_USER} — Recently Viewed Products')
print('-' * 70)
for pid in recently_viewed_products:
    t = products_pd.loc[products_pd['product_id'] == pid, 'title']
    if not t.empty:
        print(f'  {pid}: {t.values[0][:70]}')

----------------------------------------------------------------------
User 42 — Recently Viewed Products
----------------------------------------------------------------------
  P316: Mystic Fitz Roy Beige 9' 0 x 12' 0 Area Rug
  P333: Florida Shag Beige/Multi 3 ft. x 5 ft. Floral Area Rug
  P1115: Modern Gray/Multi 9 ft. x 12 ft. Vibrant Abstract Polyester Area Rug
  P1691: Genet Rust/Red-Brown 8 ft. x 11 ft. Abstract Wool Area Rug
  P1082: 1 qt. #S220-7 Molasses Extra Durable Flat Interior Paint & Primer
  P397: 1 gal. #M250-3 Apple Turnover Extra Durable Flat Interior Paint & Prim
  P1441: 1 qt. #PPU6-06 Honey Locust Eggshell Enamel Low Odor Interior Paint & 
  P1054: 1 gal. #HDPG60 Misty Emerald Lake Flat Interior Paint and Primer


In [22]:
recommendations = hybrid_recommend(TARGET_USER, recently_viewed_products)

print('-' * 70)
print(f'Top {TOP_N} Recommendations for User {TARGET_USER}')
print('-' * 70)
for _, row in recommendations.iterrows():
    print(
        f'  [{row.product_id}]  hybrid={row.hybrid_score:.3f}  '
        f'(ALS={row.als_score:.2f}, content={row.cb_score:.2f})'
    )
    print(f'    {str(row.title)[:72]}')

----------------------------------------------------------------------
Top 10 Recommendations for User 42
----------------------------------------------------------------------
  [P364]  hybrid=0.387  (ALS=0.55, content=0.00)
    1 gal. PPG1039-3 Mirror Flat Interior Paint with Primer
  [P1471]  hybrid=0.374  (ALS=0.53, content=0.00)
    5 gal. #S110-1 Secret Scent Semi-Gloss Enamel Interior Stain-Blocking Pa
  [P1417]  hybrid=0.349  (ALS=0.50, content=0.00)
    1 gal. #PPU24-01 Black Mocha Hi-Gloss Enamel Interior/Exterior Paint
  [P1717]  hybrid=0.347  (ALS=0.50, content=0.00)
    1-gal. #HDGG15U Lemon Grass Flat Latex Exterior Paint
  [P102]  hybrid=0.345  (ALS=0.49, content=0.00)
    13/16 in. x 1 in. High Speed Steel Annular Cutter with 3/4 in. Weldon Sh
  [P387]  hybrid=0.343  (ALS=0.49, content=0.00)
    1 gal. #T11-10 Wild Thing Eggshell Enamel Low Odor Interior Paint & Prim
  [P512]  hybrid=0.336  (ALS=0.48, content=0.00)
    Unfinished Alder 4 ft. L x 25 in. D x 1.5 in. T But

In [23]:
# Overlay recommended and recently viewed products on the cluster map
rec_ids = recommendations['product_id'].tolist()

def classify_point(pid):
    if pid in rec_ids:
        return 'Recommended'
    if pid in recently_viewed_products:
        return 'Recently Viewed'
    return 'Other'

pca_pd['highlight'] = pca_pd['product_id'].apply(classify_point)
pca_pd['dot_size']  = pca_pd['highlight'].map({'Recommended': 14, 'Recently Viewed': 12, 'Other': 4})

fig2 = px.scatter(
    pca_pd,
    x='x', y='y',
    color='highlight',
    size='dot_size',
    opacity=0.75,
    hover_data={'product_id': True, 'short_title': True},
    color_discrete_map={
        'Recommended':    '#e63946',
        'Recently Viewed': '#2a9d8f',
        'Other':           '#adb5bd'
    },
    title=f'Recommendations for User {TARGET_USER} — Cluster Map',
    labels={'x': 'Principal Component 1', 'y': 'Principal Component 2', 'highlight': ''}
)
fig2.update_layout(width=820, height=600)
fig2.show()

In [24]:
# Score breakdown chart: individual ALS and content scores alongside the final hybrid score
fig3 = go.Figure([
    go.Bar(
        name='ALS Score',
        x=recommendations['product_id'],
        y=recommendations['als_score'],
        marker_color='#4895ef'
    ),
    go.Bar(
        name='Content Score',
        x=recommendations['product_id'],
        y=recommendations['cb_score'],
        marker_color='#f77f00'
    ),
    go.Scatter(
        name='Hybrid Score',
        x=recommendations['product_id'],
        y=recommendations['hybrid_score'],
        mode='lines+markers',
        line=dict(color='#e63946', width=2.5),
        marker=dict(size=9)
    )
])
fig3.update_layout(
    barmode='group',
    title=f'Score Breakdown — User {TARGET_USER}  (alpha={ALPHA})',
    xaxis_title='Product ID',
    yaxis_title='Normalised Score',
    width=820,
    height=450
)
fig3.show()

In [25]:
print('Pipeline complete.')
spark.stop()

Pipeline complete.
